# Photometric Redshifts with Machine Learning

In this notebook we will study how classical machine learning methods can estimate galaxy redshifts from broad-band photometry.

We will focus on:

- supervised regression,
- feature engineering,
- selection bias,
- covariate shift,
- catastrophic outliers,
- ensemble methods,
- uncertainty proxies,
- and feature importance.

This notebook intentionally uses *toy simulations* rather than realistic survey pipelines.
The goal is pedagogical clarity rather than astrophysical realism.

Important caveat:
real photometric surveys involve:

- complex spectral energy distributions,
- non-Gaussian photometric uncertainties,
- flux-space measurements,
- selection effects,
- calibration systematics,
- and survey incompleteness.

The mock catalogue below only mimics some of these effects phenomenologically.

## 1. Setup and Mock Catalogue Generation

We generate a simplified mock galaxy catalogue.

The simulation includes:

- a redshift distribution,
- intrinsic luminosity scatter,
- approximate cosmological dimming,
- evolving galaxy colours,
- heteroscedastic photometric noise,
- and catastrophic photometric failures.

The objective is not physical realism.
Instead, we want a dataset that reproduces several qualitative properties of real photometric surveys.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from scipy.spatial.distance import pdist, squareform

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import NearestNeighbors
from astropy.cosmology import Planck18

SEED = 42
np.random.seed(SEED)

### Redshift Distribution

We generate galaxies from a smooth redshift distribution.

This is only a phenomenological approximation.
Real surveys have:

- survey selection functions,
- target incompleteness,
- cosmic variance,
- and evolving luminosity functions.

In [ ]:
# LATENT GALAXY POPULATION
N = 10_000
# Intrinsic redshift distribution
#z_spec = np.random.gamma(shape=2.0, scale=0.35, size=N)
#z_spec = np.clip(z_spec, 0.05, 2.0)
z_spec = []

while len(z_spec) < N:
    z_try = np.random.gamma(shape=2.0, scale=0.35, size=N)

    good = z_try[
        (z_try >= 0.05) &
        (z_try <= 2.0)
    ]

    z_spec.extend(good.tolist())

z_spec = np.array(z_spec[:N])

### Approximate Magnitude Model

We now generate intrinsic luminosities and apparent magnitudes.

Important:

The dimming relation below is *not* a physically accurate cosmological luminosity distance.
We only require a monotonic brightness-redshift trend for ML demonstrations.

In real analyses one would use:

- luminosity distances,
- K-corrections,
- realistic spectral templates,
- and survey throughput curves.

In [ ]:
# Simplified galaxy populations
pop_type = np.random.choice([0, 1], size=N, p=[0.6, 0.4])
# Intrinsic luminosity scatter
M_intrinsic = np.random.normal(-21.0, 1.0, size=N)

In [ ]:
# TOY OBSERVATION MODEL
#mu_toy = 9.0 * np.log10(1 + 5 * z_spec)
#i_mag_true = M_intrinsic + mu_toy + 38.5
#i_mag_true += 0.5 * np.log1p(z_spec)

d_lum = Planck18.luminosity_distance(z_spec).value  # Mpc
mu = 5 * np.log10(d_lum * 1e6 / 10)
i_mag_true = M_intrinsic + mu
i_mag_true += 0.5 * np.log1p(z_spec)


In this toy simulation the colour evolution carries more predictive information than the apparent magnitudes.

This is intentional because it allows the ML models to learn redshift-sensitive colour structure even in the presence of substantial photometric scatter.

### Smooth Colour Evolution

Galaxy colours evolve with redshift because spectral features move through the photometric filters.

Our toy model uses logistic transitions to mimic broad colour changes.
This is only phenomenological.

In real surveys, colour evolution arises from:

- stellar populations,
- dust attenuation,
- emission lines,
- and spectral-break migration.

In [ ]:
def logistic(x, x0, w):
    return 1 / (1 + np.exp(-(x - x0) / w))

In [ ]:
# Color evolution
u_g_true = 1.0 + 1.2*logistic(z_spec,0.35, 0.06) + pop_type * 0.35
g_r_true = 0.7 + 0.8*logistic(z_spec,0.75, 0.08) - 0.35*np.exp(-0.5*((z_spec-1.1)/0.18)**2) + pop_type*0.2
# Stronger colour diversity
r_i_true = 0.5 + 1.2 * logistic(z_spec, 1.15, 0.10) + 0.25 * np.random.normal(size=N)
i_z_true = 0.3 + 0.6*logistic(z_spec,1.55, 0.12)
# Intrinsic colour scatter
u_g_true += np.random.normal(0, 0.07, N)
g_r_true += np.random.normal(0, 0.07, N)
r_i_true += np.random.normal(0, 0.06, N)
i_z_true += np.random.normal(0, 0.06, N)

### Multiple Galaxy Populations

We inject two simplified overlapping galaxy populations with slightly different colour distributions.

This introduces:

- colour overlap,
- intrinsic scatter,
- and partial degeneracy.

These degeneracies are one of the main reasons photometric redshifts are difficult.

In [ ]:
# Add population variance (e.g., Red vs Blue galaxies)
u_g_true += pop_type * 0.4 + np.random.normal(0, 0.05, N)
g_r_true += pop_type * 0.2 + np.random.normal(0, 0.05, N)
# Construct magnitudes
i_true = i_mag_true
r_true = i_true + r_i_true
g_true = r_true + g_r_true
u_true = g_true + u_g_true
z_mag_true = i_true - i_z_true

### Photometric Noise

We now inject magnitude-dependent photometric uncertainty.

This creates heteroscedastic noise:

- bright galaxies have small uncertainties,
- faint galaxies have larger uncertainties.

Important caveat:

Real survey pipelines usually operate in flux space. Near detection limits, magnitude uncertainties become highly non-Gaussian.

Real photometric pipelines can also introduce correlated uncertainties between bands due to calibration errors, PSF modelling, aperture definitions, and shared background estimation.


In [ ]:
# Magnitude-dependent noise (Heteroscedastic)
#sigma = np.clip(0.02 + 0.03 * (i_true - 20.0)**2, 0.02, 0.4)
# Heteroscedastic magnitude uncertainties.
# Noise rises toward the faint end but remains bounded.
#sigma = 0.02 * np.exp(0.25 * (i_true - 20))
#sigma = np.clip(sigma, 0.02, 0.3)
sigma = 0.02 + 0.015 * np.clip(i_true - 20, 0, None)**1.5
sigma = np.clip(sigma, 0.02, 0.25)

z_mag = z_mag_true + np.random.normal(0, sigma)
u = u_true + np.random.normal(0, sigma)
g = g_true + np.random.normal(0, sigma)
r = r_true + np.random.normal(0, sigma)
i = i_true + np.random.normal(0, sigma)


### Catastrophic Photometric Failures

Real surveys contain:

- blending,
- calibration problems,
- bad deblending,
- cosmic rays,
- and failed measurements.

We inject a small population of catastrophic photometric errors.

In [ ]:
# Inject catastrophic photometric failures
#
# These mimic occasional survey problems such as:
# - bad deblending,
# - corrupted apertures,
# - cosmic rays,
# - saturation,
# - or calibration failures.

cat_idx = np.random.choice(
    np.arange(N),
    size=int(0.03 * N),
    replace=False
)
bands = [u, g, r, i, z_mag]
for band in bands:
    # Only some catastrophic objects fail in a given band
    corrupt = np.random.rand(len(cat_idx)) < 0.5
    # Large non-Gaussian perturbations
    # Mixture distribution: most failures are moderate, a smaller fraction are extreme.
    offsets = np.where(
        np.random.rand(len(cat_idx)) < 0.8,
        np.random.normal(0, 0.35, size=len(cat_idx)),
        np.random.normal(0, 1.0, size=len(cat_idx))
    )
    band[cat_idx[corrupt]] += offsets[corrupt]

In [ ]:
# Assemble DataFrame
df = pd.DataFrame({
    'u': u, 'g': g, 'r': r, 'i': i, 'z_mag': z_mag,
    'z_spec': z_spec, 'sigma': sigma
})

# Feature Engineering
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z_mag']

### Important Note: Feature Redundancy and Multicollinearity

Our feature set deliberately mixes relative spectral shape (the colors) with overall brightness (the magnitudes). Because colors are simply magnitude differences, this introduces strong algebraic redundancy.

Given a single reference magnitude and the colors, all other magnitudes can be reconstructed exactly. For example:


$$g = r + (g-r)$$

$$i = r - (r-i)$$

As a result, subsets like (`r`, `i`, `r_i`) are perfectly linearly dependent. While this strict multicollinearity does not prevent most machine learning models from fitting the data successfully, it significantly complicates:

* **Coefficient interpretation**
* **Feature-importance analysis**
* **Attribution methods**

This design intentionally mirrors real astronomical ML pipelines, where engineered color features inherently remain highly correlated with the source magnitudes.

### Feature Correlations

Our toy catalogue contains highly correlated features.

This is realistic qualitatively, although the exact algebraic structure here is simplified.

Strong feature correlations are important because:

- they affect feature importance,
- they affect interpretability,
- and they can hide information redundancy.

In [ ]:
corr = df[["u_g", "g_r", "r_i", "i_z", "r", "i"]].corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(df["z_spec"], df["g_r"], s=6, alpha=0.2)
plt.xlabel("Spectroscopic redshift")
plt.ylabel("g - r")
plt.title("Colour Evolution with Redshift")
plt.tight_layout()
plt.show()

## 2. Spectroscopic Selection Bias and Covariate Shift

Machine learning models are only as good as their training data.

In astronomy, spectroscopic samples are rarely representative of the full photometric survey.
Bright galaxies are generally easier to observe spectroscopically than faint galaxies.

Because apparent magnitude correlates with redshift in flux-limited surveys, this selection also indirectly alters the redshift distribution and the local feature-space coverage of the training sample.

Important distinction:

In ideal covariate shift:

$P_{\rm train}(x) \neq P_{\rm test}(x)$

while:

$P(y|x)_{\rm train} = P(y|x)_{\rm test}$

Our simulation does not satisfy pure covariate shift.

The observed photometric features are noisy measurements whose uncertainty increases toward faint magnitudes.

As a result, the conditional relation

$ P(z \mid x_{\mathrm{obs}}) $

also changes with survey depth because the observed feature vectors become progressively noisier and more broadly distributed.

That is our setup approximates covariate shift, but does not strictly satisfy its assumptions.

Because photometric noise increases with magnitude and catastrophic failures are injected non-uniformly, the conditional distribution ( $P(y|x)$ ) is also mildly perturbed in observed feature space.

The result is better described as a mixture of:
 * covariate shift,
 * heteroscedastic measurement corruption,
 * support mismatch,
 * and mild conditional-distribution shift.


In [ ]:
features = ["u_g", "g_r", "r_i", "i_z", "r", "i"]
X = df[features].copy()
y = df["z_spec"].copy()
# Probability of getting a spectrum drops as galaxies get fainter
#p_spec = 1 / (1 + np.exp(1.5 * (df['r'] - 21.5)))
#p_spec = 1 / (1 + np.exp(1.0 * (df['r'] - 22.5)))
p_spec = 1 / (1 + np.exp(1.2 * (df['r'] - 22.5)))
p_spec = np.clip(p_spec, 0.05, 0.9)
has_spec = np.random.rand(len(df)) < p_spec

### Important Note About Evaluation

In real surveys, the deployment distribution is unlabeled.

Here we keep labels for the photometric sample only because this is an educational simulation.
This allows us to measure how badly models fail under distribution shift.

###### Important pedagogical note:

In this toy simulation we generate the full mock catalogue before defining
training and deployment samples.

In real ML pipelines, all preprocessing operations that learn from the data
(e.g. normalization, imputation, feature selection, PCA, target encoding)
must be fit only on the training set to avoid data leakage.

The engineered colour features here are deterministic algebraic combinations of observed magnitudes and therefore do not estimate any global statistics from the full dataset.

In contrast, operations such as:

- normalization,
- PCA,
- imputation,
- target encoding,
- feature selection,
- or learned embeddings

must always be fit using training data only.

In [ ]:
X_spec = df.loc[has_spec, features]
y_spec = df.loc[has_spec, 'z_spec']
X_photo = df.loc[~has_spec, features]
y_photo = df.loc[~has_spec, 'z_spec']
X_train, X_val, y_train, y_val = train_test_split(X_spec, y_spec, test_size=0.2, random_state=SEED)
print(f"Training on {len(X_train)} bright galaxies.")
print(f"Testing on {len(X_photo)} faint galaxies (Covariate Shift).")
X_deploy = X_photo
y_deploy = y_photo
bright_limit = 21.0
bright_mask_val = X_val["r"] <= bright_limit
faint_mask_val = X_val["r"] > bright_limit
bright_mask_deploy = X_deploy["r"] <= bright_limit
faint_mask_deploy = X_deploy["r"] > bright_limit
print(f"Training size:   {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size:       {len(X_deploy)}")
plt.hist(df["r"], bins=35, alpha=0.6, label="Full survey")
plt.hist(df.loc[has_spec, "r"], bins=35, alpha=0.6, label="Spectroscopic sample")
plt.axvline(21.0, ls="--", color="black")
plt.xlabel("r magnitude")
plt.ylabel("Count")
plt.yscale("log")
plt.title("Selection Bias in the Spectroscopic Sample")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hexbin(X_train["g_r"], X_train["r_i"], gridsize=40, mincnt=1)
axes[0].set_title("Training Distribution")
axes[0].set_xlabel("g - r")
axes[0].set_ylabel("r - i")
axes[1].hexbin(X_deploy["g_r"], X_deploy["r_i"], gridsize=40, mincnt=1)
axes[1].set_title("Deployment Distribution")
axes[1].set_xlabel("g - r")
axes[1].set_ylabel("r - i")
plt.tight_layout()
plt.show()

## 3. Model Training and Evaluation

We compare:

- a trivial baseline,
- regularized linear regression (Ridge),
- random forests,
- and gradient boosting.

###### Important 1:

Tree-based methods are largely insensitive to feature scaling because they split on thresholds.
Linear models, however, often benefit from scaling.

Notice that we still could place tree models inside pipelines for consistency and reproducibility, even if scaling itself is unnecessary.

###### Important 2:

Tree ensembles partition the feature space into regions defined by the training data.
Outside the training support, tree ensembles typically continue predicting using terminal regions learned from the nearest available training examples rather than inferring new physical trends.

They interpolate well within the training domain but extrapolate poorly beyond it.

This becomes especially important in astronomy where deployment data may extend beyond the spectroscopic training coverage.


###### Note on multicollinearity:

Recall that since our feature set includes $r$, $i$, and $\text{r_i} = r - i$ we have exact linear dependencies among subsets of the engineered features.

Ordinary least squares regression predictions remain well-defined through pseudoinverse/SVD-based solvers, but the coefficients themselves become non-identifiable and unstable.

Ridge regression adds L2 regularization, which stabilizes the solution by penalizing large coefficients.

Even with Ridge regularization, coefficient interpretation remains difficult because strongly correlated or algebraically dependent features can redistribute weight between one another without changing predictions substantially.
 
Tree-based models are much less sensitive to multicollinearity because they do not invert a covariance matrix.

In [ ]:
models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Ridge Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.7,
        random_state=SEED
    )
}

### Why scaling only the linear model?

Linear models are sensitive to feature scales because the regression coefficients depend directly on the numerical magnitude of the inputs.

Tree-based methods are largely insensitive to feature scaling because they split on thresholds rather than distances.

### Photometric redshift metrics

In photometric-redshift analyses we usually define normalized residuals as

$\Delta z = \frac{ z_{\mathrm{pred}} - z_{\mathrm{true}}}{1 + z_{\mathrm{true}}}$

Common evaluation metrics include:

- RMS Normalized,
- bias,
- NMAD,
- and catastrophic outlier fraction.

Throughout this notebook we use:

- median normalized residual as the bias estimator,
- NMAD as a robust scatter estimator,
- and `RMSE_NORM` as a non-robust global error metric.

NMAD is preferred in photometric-redshift analyses because it is less sensitive to catastrophic outliers than standard deviation or `RMSE_NORM`.

The factor 1.48 rescales the median absolute deviation so that it equals the standard deviation for a Gaussian distribution.

###### Outliers
We can consider Outliers as the points with:

$|\Delta z| > 0.15$

where the exact threshold varies across surveys and is partly historical rather than physically unique.


In [ ]:
def get_metrics(y_true, y_pred, label="Dataset"):
    dz = (y_pred - y_true) / (1 + y_true)
    nmad = 1.48 * np.median(np.abs(dz - np.median(dz)))
    outlier_frac = np.mean(np.abs(dz) > 0.15)
    rmse_norm = np.sqrt(np.mean(dz**2))
    return {"Set": label, "RMSE_NORM": rmse_norm, "NMAD": nmad, "Outlier%": outlier_frac * 100}

In [ ]:
predictions_val = {}
predictions_deploy = {}
# Training and Evaluating
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    pred_deploy = model.predict(X_deploy)
    predictions_val[name] = pred_val
    predictions_deploy[name] = pred_deploy
    results.append(get_metrics(y_val, pred_val, f"{name} (Val)"))
    results.append(get_metrics(y_photo, pred_deploy, f"{name} (Deployment)"))
display(pd.DataFrame(results).round(4))

# 4. Overfitting and Learning Curves

A central question in machine learning is whether a model is:

- underfitting,
- well-generalized,
- or overfitting.

Learning curves are one of the most informative diagnostics.

In [ ]:
def normalized_rmse(y_true, y_pred):
    dz = (y_pred - y_true) / (1 + y_true)
    return np.sqrt(np.mean(dz**2))

Cross-validation estimates here remain internally consistent because all folds are drawn from the same spectroscopic-selection process.

However, this does not guarantee robustness under deployment onto the deeper photometric population.

In [ ]:
train_fracs = np.linspace(0.1, 1.0, 8)

train_errors = []
val_errors_same_dist = []
deploy_errors_shifted = []

for frac in train_fracs:
    n_train = int(frac * len(X_train))
    subset_idx = np.random.choice(len(X_train), n_train, replace=False)
    X_sub = X_train.iloc[subset_idx]
    y_sub = y_train.iloc[subset_idx]
    gb_fracs = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.7,
        random_state=SEED
    )

    gb_fracs.fit(X_sub, y_sub)
    pred_train = gb_fracs.predict(X_sub)
    pred_val = gb_fracs.predict(X_val)
    pred_deploy = gb_fracs.predict(X_deploy)
    train_errors.append(normalized_rmse(y_sub, pred_train))
    val_errors_same_dist.append(normalized_rmse(y_val, pred_val))
    deploy_errors_shifted.append(normalized_rmse(y_deploy, pred_deploy))

plt.figure(figsize=(7,5))
plt.plot(train_fracs, train_errors, marker='o', label='Train')
plt.plot(train_fracs, val_errors_same_dist, marker='o', label='Validation')
plt.plot(train_fracs, deploy_errors_shifted, marker='o', label='Deployment')
plt.xlabel('Fraction of spectroscopic training set used')
plt.ylabel('RMSE_NORM')
plt.title('Learning Curves Under Distribution Shift')
plt.legend()
plt.tight_layout()
plt.show()

# 5. Catastrophic Outliers

A common convention defines catastrophic outliers as

$|\Delta z| > 0.15$

The exact threshold varies across surveys and is partly historical rather than physically unique.

These failures often arise from:

- colour degeneracies,
- noisy photometry,
- sparse training coverage,
- and extrapolation outside the training domain.

In [ ]:
best = "Gradient Boosting"
best_pred_deploy = predictions_deploy[best]
dz = (best_pred_deploy - y_deploy) / (1 + y_deploy)
cat_mask = np.abs(dz) > 0.15
plt.figure(figsize=(6, 6))
plt.scatter(
    y_deploy[~cat_mask],
    best_pred_deploy[~cat_mask],
    s=5,
    alpha=0.15,
    rasterized=True,
    label="Normal"
)
plt.scatter(
    y_deploy[cat_mask],
    best_pred_deploy[cat_mask],
    s=12,
    alpha=0.7,
    rasterized=True,
    label="Catastrophic"
)
plt.plot([0, 2], [0, 2], "r--", lw=1)
plt.xlim(0, 2)
plt.ylim(0, 2)
plt.xlabel("True redshift")
plt.ylabel("Predicted redshift")
plt.title("Catastrophic Photo-z Failures")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
g = sns.jointplot(
    x=y_deploy,
    y=best_pred_deploy,
    kind="hex",
    height=7,
    joint_kws={"bins": "log"}
)
g.ax_joint.plot([0, 2], [0, 2], "r--", lw=1)
g.ax_joint.set_xlim(0, 2)
g.ax_joint.set_ylim(0, 2)
g.ax_joint.set_xlabel("True redshift")
g.ax_joint.set_ylabel("Predicted redshift")
g.fig.suptitle("True vs Predicted Redshift", y=1.02)
plt.show()

Notice how predictions begin collapsing toward the population mean at high redshift.

This is a common regression behavior:

- noisy regions shrink toward the mean prediction,
- sparse regions become poorly constrained,
- and tree ensembles extrapolate poorly beyond the training support.

In [ ]:
residuals = (best_pred_deploy - y_deploy) / (1 + y_deploy)
g = sns.jointplot(x=y_deploy, y=residuals, kind="hex", height=7)
g.ax_joint.axhline(0, color="red", ls="--", lw=1)
g.ax_joint.set_xlabel("True redshift")
g.ax_joint.set_ylabel("Normalized Residual Δz")
g.ax_joint.set_xlim(0, 2)
lim = np.percentile(np.abs(residuals), 99)
g.ax_joint.set_ylim(-lim, lim)
g.fig.suptitle("Residual Structure", y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(dz, bins=80)
plt.axvline(0, color="black", ls="--")
plt.xlabel("Normalized residual Δz")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()

### Sensitivity to feature-space shift / photometric zeropoint drift

Let's simulate a **systematic calibration error** (zero-point shift) by injecting a $0.03$ magnitude offset into the $g-r$ color of the deployment set. 

We evaluate how this small physical bias propagates into significant photometric redshift errors, demonstrating the model's sensitivity to mismatches between training and survey data.


In [ ]:
gb = models["Gradient Boosting"]
# Deployment survey calibration mismatch
X_deploy_shifted = X_deploy.copy()
X_deploy_shifted['g_r'] += 0.03
pred_shifted = gb.predict(X_deploy_shifted)

display(
    pd.DataFrame([
        get_metrics(y_deploy, best_pred_deploy, 'Original deployment'),
        get_metrics(y_deploy, pred_shifted, 'Calibration shifted')
    ])
)

# 6. Error Surfaces in Parameter Space

Model performance is not uniform across parameter space.

The prediction quality depends strongly on:

- redshift,
- magnitude,
- photometric noise,
- and local feature-space density.

We now examine how the error varies across the $(z, r)$ plane.

**Interpretation**:

- Errors increase rapidly at faint magnitudes.
- Sparse regions of feature space produce unstable predictions.
- Catastrophic outliers cluster where the training coverage becomes poor.
- The object-count panel is important because low-density bins can produce noisy metrics.


In [ ]:
z_bins = np.linspace(0, 2, 10)
r_bins = np.linspace(X_deploy["r"].min(), X_deploy["r"].max(), 10)
z_cent = 0.5 * (z_bins[:-1] + z_bins[1:])
r_cent = 0.5 * (r_bins[:-1] + r_bins[1:])
# Initialize the grids
RMSE_NORM = np.full((len(z_cent), len(r_cent)), np.nan)
BIAS = np.full_like(RMSE_NORM, np.nan)
OUT = np.full_like(RMSE_NORM, np.nan)
COUNTS = np.full_like(RMSE_NORM, np.nan)

for i in range(len(z_bins) - 1):
    for j in range(len(r_bins) - 1):
        m = (
            (y_deploy >= z_bins[i]) &
            (y_deploy < z_bins[i + 1]) &
            (X_deploy["r"] >= r_bins[j]) &
            (X_deploy["r"] < r_bins[j + 1])
        )

        if m.sum() > 30:
            # Normalized residuals: (pred - true) / (1 + true)
            dz_local = (best_pred_deploy[m] - y_deploy[m]) / (1 + y_deploy[m])
            
            RMSE_NORM[i, j] = np.sqrt(np.mean(dz_local**2))
            BIAS[i, j] = np.median(dz_local)
            OUT[i, j] = np.mean(np.abs(dz_local) > 0.15)
            COUNTS[i, j] = m.sum()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)
im0 = axes[0].imshow(
    RMSE_NORM,
    origin="lower",
    aspect="auto",
    extent=[
        r_cent.min(),
        r_cent.max(),
        z_cent.min(),
        z_cent.max()
    ],
    cmap="viridis"
)
axes[0].set_title("RMSE_NORM")
axes[0].set_xlabel("r magnitude")
axes[0].set_ylabel("Redshift")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(
    BIAS,
    origin="lower",
    aspect="auto",
    extent=[
        r_cent.min(),
        r_cent.max(),
        z_cent.min(),
        z_cent.max()
    ],
    cmap="coolwarm"
)
axes[1].set_title("Bias")
axes[1].set_xlabel("r magnitude")
axes[1].set_ylabel("Redshift")
plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(
    OUT,
    origin="lower",
    aspect="auto",
    extent=[
        r_cent.min(),
        r_cent.max(),
        z_cent.min(),
        z_cent.max()
    ],
    cmap="magma"
)
axes[2].set_title("Outlier Fraction")
axes[2].set_xlabel("r magnitude")
axes[2].set_ylabel("Redshift")
plt.colorbar(im2, ax=axes[2])
im3 = axes[3].imshow(
    COUNTS,
    origin="lower",
    aspect="auto",
    extent=[
        r_cent.min(),
        r_cent.max(),
        z_cent.min(),
        z_cent.max()
    ],
    cmap="cividis"
)
axes[3].set_title("Objects per Bin")
axes[3].set_xlabel("r magnitude")
axes[3].set_ylabel("Redshift")
plt.colorbar(im3, ax=axes[3])
fig.suptitle("Photometric Redshift Error Surfaces", fontsize=14)
plt.show()

# 7. Feature Reliance Under Distribution Shift

We now examine how feature importance changes between:

- the spectroscopic training distribution,
- and the deployment distribution.

Important caveat:

- Permutation importance becomes difficult to interpret when features are strongly correlated.
If information is redundant, the apparent importance of individual variables may be underestimated.

- Permutation importance estimates can occasionally become negative due to finite-sample noise or strong feature correlations.

> A slightly negative importance does not imply that a feature is physically anti-predictive.
Instead, it usually indicates that the feature contributes little unique information beyond correlated variables.

In [ ]:
gb = models["Gradient Boosting"]
perm_spec = permutation_importance(
    gb,
    X_val,
    y_val,
    n_repeats=15,
    random_state=SEED,
    n_jobs=-1
)
perm_full = permutation_importance(
    gb,
    X_deploy,
    y_deploy,
    n_repeats=15,
    random_state=SEED,
    n_jobs=-1
)
features = X_val.columns
imp_spec = pd.Series(perm_spec.importances_mean, index=features)
imp_full = pd.Series(perm_full.importances_mean, index=features)
# Normalize by total absolute importance to reduce instability from correlated negative values.
norm_spec = np.abs(imp_spec).sum()
norm_full = np.abs(imp_full).sum()

if norm_spec > 0:
    imp_spec = imp_spec / norm_spec

if norm_full > 0:
    imp_full = imp_full / norm_full
    

df_imp = pd.DataFrame({ "Spectroscopic": imp_spec, "Deployment": imp_full})
df_imp["Shift"] = (df_imp["Deployment"] - df_imp["Spectroscopic"])
df_imp = df_imp.sort_values("Spectroscopic")

x = np.arange(len(df_imp.index))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
axes[0].barh(x - 0.2, df_imp["Spectroscopic"], height=0.4, label="Spectroscopic")
axes[0].barh(x + 0.2, df_imp["Deployment"], height=0.4, label="Deployment")
axes[0].set_yticks(x)
axes[0].set_yticklabels(df_imp.index)
axes[0].set_xlabel("Normalized Importance")
axes[0].set_title("Feature Reliance")
axes[0].legend()
axes[1].barh(x, df_imp["Shift"])
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_yticks(x)
axes[1].set_yticklabels(df_imp.index)
axes[1].set_xlabel("Importance Shift")
axes[1].set_title("Distribution-Dependent Changes")
plt.suptitle("Feature Reliance Under Covariate Shift", fontsize=14)
plt.tight_layout()
plt.show()

Interpretation:

The trained model itself is fixed after training.

However, permutation importance is distribution-dependent.
A feature can appear more or less important depending on:

- which regions of feature space are populated,
- how correlated the features are,
- and where the model predictions are most sensitive locally.

Under covariate shift, the apparent feature importance may therefore change
even when the underlying learned mapping is unchanged.

This does NOT necessarily imply changing physics.

Instead, the apparent feature importance can change because:

- the feature distribution changes,
- the correlation structure changes,
- sparse regions become more important,
- and extrapolation behavior changes.

In [ ]:
support_pipeline = make_pipeline(StandardScaler(), NearestNeighbors(n_neighbors=5))
support_pipeline.fit(X_train)
scaled_X_deploy = support_pipeline.named_steps['standardscaler'].transform(X_deploy)
nn_model = support_pipeline.named_steps['nearestneighbors']
distances, _ = nn_model.kneighbors(scaled_X_deploy)
support_distance = distances.mean(axis=1)
plt.figure(figsize=(8, 6))
hb = plt.hexbin(
    support_distance, 
    np.abs(residuals), 
    gridsize=35, 
    bins='log', 
    cmap='viridis'
)
plt.colorbar(hb, label='log10(N)')
plt.xlabel("Distance to Training Support (Feature Space)")
plt.ylabel("Absolute Normalized Residual |Δz|")
plt.title("Error vs. Feature Space Extrapolation")
plt.tight_layout()
plt.show()

Important observation:

Tree ensembles usually interpolate well within regions of feature space that are densely represented in the training data.

Performance degrades rapidly in sparse or weakly represented regions, and true extrapolation beyond the training-domain support is usually very poor.

Predictions beyond the spectroscopic training coverage often collapse toward the boundary of the training distribution rather than continuing the true physical trend.

The relevant notion of extrapolation occurs in the multidimensional feature space rather than only along the redshift axis.

Even galaxies whose true redshifts lie inside the training redshift range may occupy regions of colour space that are poorly represented in the spectroscopic sample.


In [ ]:
# Restrict training to lower redshift galaxies only
mask_lowz = y_spec < 1.0
X_train_extra = X_spec.loc[mask_lowz]
y_train_extra = y_spec.loc[mask_lowz]
gb_extra = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.7,
    random_state=SEED
)

gb_extra.fit(X_train_extra, y_train_extra)
pred_extra = gb_extra.predict(X_photo)
plt.figure(figsize=(6, 6))
plt.scatter(y_photo, pred_extra, s=5, alpha=0.15)
plt.axvline(1.0, color="red", ls="--", label="Training limit")
plt.axhline(1.0, color="red", ls="--")
plt.plot([0, 2], [0, 2], "k--")
plt.xlabel("True redshift")
plt.ylabel("Predicted redshift")
plt.title("Failure Outside the Effective Training Support")
plt.legend()
plt.tight_layout()
plt.show()

# 8. Ensemble Disagreement as a Heuristic Uncertainty Proxy

**Ensemble Spread as an Uncertainty Proxy**

Random Forests use bootstrap resampling and randomized feature selection to reduce correlations between trees.
While the spread of predictions across these trees is often used as a heuristic indicator of uncertainty, it merely measures the model's sensitivity to resampling—not a formally calibrated Bayesian posterior.

**The Conflation of Errors**

Separating uncertainty robustly is an active research problem. Currently, ensemble spread conflates three distinct effects:

* **Epistemic uncertainty:** Sparse training coverage and local ambiguity.
* **Aleatoric uncertainty:** Intrinsic observational (heteroscedastic) noise.
* **Model instability:** Variance stemming from randomized tree construction.

**Low variance does not imply correctness.**

A critical failure mode occurs under **distribution shift**. In poorly represented regions of feature space, all trees in the ensemble tend to make correlated extrapolation errors because they share the same underlying training data and inductive biases.

Consequently, the ensemble standard deviation can remain artificially small even when predictions are systematically wrong. While ensemble disagreement often correlates with prediction difficulty, it can substantially underestimate the true error and should never be interpreted as reliable confidence.

The ensemble spread is also sensitive to hyperparameters such as:

- tree depth,
- bootstrap fraction,
- feature subsampling,
- and number of trees.

It is therefore not an intrinsic property of the underlying data-generating process.

In [ ]:
rf = models["Random Forest"]
gr_min, gr_max = (X_deploy["g_r"].min(), X_deploy["g_r"].max())
ri_min, ri_max = (X_deploy["r_i"].min(), X_deploy["r_i"].max())
gr_grid = np.linspace(gr_min, gr_max, 100)
ri_grid = np.linspace(ri_min, ri_max, 100)
GR, RI = np.meshgrid(gr_grid, ri_grid)

We visualize a 2D slice through the higher-dimensional feature space.

Important:

Many combinations in this slice are not physically populated galaxies.

This visualization should therefore be interpreted geometrically rather than physically.

In [ ]:
X_grid = pd.DataFrame({
    "g_r": GR.ravel(),
    "r_i": RI.ravel(),
    "u_g": X_train["u_g"].median(),
    "i_z": X_train["i_z"].median(),
    "r": X_train["r"].median(),
    "i": X_train["r"].median() - RI.ravel()
})

X_grid = X_grid[X_train.columns]
Z_mean = rf.predict(X_grid).reshape(GR.shape)
# Suppress sklearn feature name warnings. The base DecisionTreeRegressors 
# expect unlabelled arrays, but complain if the parent RF had feature names.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    tree_preds = np.vstack([
        tree.predict(X_grid.values)
        for tree in rf.estimators_
    ])

Z_std = tree_preds.std(axis=0).reshape(GR.shape)
fig, ax = plt.subplots(figsize=(8, 6))
cmap = ax.contourf(GR, RI, Z_mean, levels=40, cmap="viridis", alpha=0.9)
cbar = plt.colorbar(cmap, ax=ax)
cbar.set_label("Predicted Redshift")
unc = ax.contourf(GR, RI, Z_std, levels=20, cmap="Reds", alpha=0.25)
cbar2 = plt.colorbar(unc, ax=ax)
cbar2.set_label("Ensemble Spread")

ax.scatter(
    X_deploy["g_r"],
    X_deploy["r_i"],
    s=2,
    alpha=0.15,
    color="white",
    label="Observed galaxies"
)
ax.set_xlabel("g - r")
ax.set_ylabel("r - i")
ax.set_title("Model Geometry in Colour Space")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

Random-forest ensemble variance is not a statistically rigorous predictive uncertainty.

Because the trees are trained on highly overlapping bootstrap samples and share the same training distribution, the ensemble members are algorithmically correlated bootstrap estimators rather than samples from a Bayesian posterior predictive distribution.

In [ ]:
tree_preds = np.array([tree.predict(X_photo.values) for tree in rf.estimators_])
std_devs = tree_preds.std(axis=0)
std_devs_norm = std_devs / (1 + y_photo.values)
errors_norm = np.abs((rf.predict(X_photo) - y_photo.values) / (1 + y_photo.values))
#errors = np.abs((rf.predict(X_photo) - y_photo.values)/ (1 + y_photo.values))
plt.figure(figsize=(8, 5))
plt.hexbin(std_devs_norm, errors_norm, gridsize=30, cmap='Blues', bins='log')
plt.xlabel("Ensemble Spread (Heuristic Uncertainty)")
plt.ylabel("Absolute Prediction Error")
plt.title("Correlation between Model Disagreement and Error")
plt.colorbar(label='log10(N)')
plt.show()

Interpretation:

Ensemble disagreement correlates with prediction difficulty, but only imperfectly.

This is not a calibrated uncertainty estimate.

In real scientific analyses, uncertainty calibration is a major challenge.

An important limitation is that ensemble disagreement can become severely miscalibrated under covariate shift.

A model may produce artificially confident predictions in regions of feature space that are poorly represented in the training set if all ensemble members make similar extrapolation errors.


# 9. Catastrophic Outliers and Colour Degeneracies

In real surveys, catastrophic failures often arise from spectral-feature degeneracies.

For example:

- Balmer breaks,
- and Lyman breaks,

can produce similar broad-band colours at different redshifts.

Our toy simulation does not model spectra explicitly.
However, overlapping colour distributions still create phenomenological degeneracies.

In [ ]:
colour_cols = ["u_g", "g_r", "r_i", "i_z"]
df_s = df.sample(500, random_state=SEED)
X_cols = df_s[colour_cols].to_numpy()
z_vals = df_s["z_spec"].to_numpy()
idxs = df_s.index.to_numpy()
cov = np.cov(X_cols, rowvar=False)
cov += 1e-6 * np.eye(cov.shape[0])
inv_cov = np.linalg.pinv(cov)

Mahalanobis distance accounts for correlated feature geometry.

Unlike Euclidean distance, it measures similarity relative to the covariance structure of the data.

In [ ]:
# 1. Calculate Mahalanobis distance between all pairs in the sub-sample.
# This metric accounts for the correlated covariance structure of the colour distribution.
# of galaxy colors (the "stellar locus").
mahal_dist = squareform(pdist(X_cols, metric="mahalanobis", VI=inv_cov))
# 2. Compute the redshift separation between all pairs.
dz_matrix = np.abs(z_vals[:, None] - z_vals[None, :])
# 3. Define the degeneracy criteria.
# We look for pairs with a significant redshift difference (delta_z > 0.5)
# to simulate "catastrophic" confusion.
z_sep_threshold = 0.5
mask = dz_matrix > z_sep_threshold
if not np.any(mask):
    print(f"No pairs found with delta_z > {z_sep_threshold}. Increase sample size or lower threshold.")
else:
    # Find the pair with the smallest Mahalanobis color distance 
    # among those with large redshift separation.
    masked_dist = np.where(mask, mahal_dist, np.inf)
    i, j = np.unravel_index(np.argmin(masked_dist), masked_dist.shape)
    # Retrieve original dataframe indices
    idx_a, idx_b = idxs[i], idxs[j]
    print(f"--- PHOTOMETRIC DEGENERACY FOUND ---")
    print(f"Galaxy A: z_spec = {df.loc[idx_a, 'z_spec']:.3f}, r_mag = {df.loc[idx_a, 'r']:.2f}")
    print(f"Galaxy B: z_spec = {df.loc[idx_b, 'z_spec']:.3f}, r_mag = {df.loc[idx_b, 'r']:.2f}")
    print(f"Redshift Separation (Δz): {dz_matrix[i, j]:.3f}")
    print(f"Mahalanobis Color Distance: {mahal_dist[i, j]:.5f}")
    print("-" * 40)
    # Display color features for comparison
    display(df.loc[[idx_a, idx_b], colour_cols])
    # 4. Visualization: Show the degeneracy in color-color space.
    plt.figure(figsize=(8, 6))
    # Plot the background distribution
    plt.scatter(df_s['g_r'], df_s['r_i'], c='lightgray', alpha=0.3, s=15, label='Sample Population')
    # Highlight the degenerate pair
    plt.scatter(df.loc[idx_a, 'g_r'], df.loc[idx_a, 'r_i'], 
                color='firebrick', s=120, marker='X', edgecolors='black', 
                label=f'Galaxy A (z={df.loc[idx_a, "z_spec"]:.2f})')
    
    plt.scatter(df.loc[idx_b, 'g_r'], df.loc[idx_b, 'r_i'], 
                color='royalblue', s=120, marker='o', edgecolors='black', 
                label=f'Galaxy B (z={df.loc[idx_b, "z_spec"]:.2f})')

    plt.xlabel("g - r")
    plt.ylabel("r - i")
    plt.title("Colour Degeneracy in Feature Space")
    plt.legend(frameon=True, facecolor='white')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

These two galaxies are approximately degenerate within the limited colour feature space provided to the ML model.

This does not imply that the underlying galaxies are physically identical.

Additional information such as spectroscopy, morphology, narrow-band photometry, or higher-S/N measurements could break the degeneracy.

**Important caveat**:

Mahalanobis distance only measures similarity within the chosen feature representation.

Two galaxies that appear nearly identical in this limited colour space may become distinguishable once additional information is included, such as:

- morphology,
- spectroscopy,
- narrow-band photometry,
- variability,
- or higher signal-to-noise observations.

**Feature-space degeneracy is therefore representation-dependent**.

In sufficiently large samples, nearest-neighbour searches in noisy low-dimensional spaces will almost always produce apparently degenerate pairs.

In [ ]:
bins = np.linspace(0, 2, 20)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bias_z = []
scatter_z = []
for i in range(len(bins) - 1):
    m = ((y_deploy >= bins[i]) & (y_deploy < bins[i + 1]))
    if m.sum() > 20:
        dz_bin = dz[m]
        bias_z.append(np.median(dz_bin))
        scatter_z.append(1.48 * np.median(np.abs(dz_bin - np.median(dz_bin))))
    else:
        bias_z.append(np.nan)
        scatter_z.append(np.nan)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bin_centers, bias_z, marker="o", label="Bias")
ax.plot(bin_centers, scatter_z, marker="o",label="NMAD")
ax.axhline(0, color="black", ls="--")
ax.set_xlabel("True redshift")
ax.set_ylabel("Metric value")
ax.set_title("Redshift-Dependent Performance")
ax.legend()
plt.tight_layout()
plt.show()

An important limitation of this toy simulation is that the colour-redshift relation is intentionally smooth and low-dimensional.

Real galaxy populations exhibit substantially more complexity:

- broader intrinsic population diversity,
- dust reddening,
- non-monotonic spectral-feature migration,
- emission-line contamination,
- and stronger colour degeneracies.

As a result, real photometric-redshift estimation is typically much more difficult than this demonstration suggests.

# 10. Reliability of Uncertainty Proxies

A useful uncertainty estimate should be calibrated.

For example, among galaxies assigned a nominal 68% confidence interval, roughly 68% should contain the true redshift.

Many ML uncertainty proxies fail this criterion under distribution shift.

A calibrated uncertainty estimator would produce a predictable relationship between predicted uncertainty and empirical error.

The relation here is only approximate.

The ensemble spread captures some prediction difficulty but remains poorly calibrated, especially under distribution shift.


In [ ]:
rf_pred = rf.predict(X_photo)
tree_preds = np.array([tree.predict(X_photo.values) for tree in rf.estimators_])
rf_std = tree_preds.std(axis=0)
abs_err = np.abs((rf_pred - y_photo.values) / (1 + y_photo.values))
bins = np.unique(np.quantile(rf_std, np.linspace(0, 1, 8)))
bin_centers = []
empirical_err = []
for i in range(len(bins)-1):
    m = ((rf_std >= bins[i]) & (rf_std < bins[i+1]))
    if m.sum() > 20:
        bin_centers.append(np.median(rf_std[m]))
        empirical_err.append(np.median(abs_err[m]))

plt.figure(figsize=(6,5))
plt.plot(bin_centers, empirical_err, marker="o")
plt.xlabel("Predicted Ensemble Spread")
plt.ylabel("Median Empirical Error")
plt.title("Uncertainty Proxy Calibration")
plt.tight_layout()
plt.show()

Modern photometric-redshift pipelines often attempt to mitigate spectroscopic-selection bias using:

- importance weighting,
- domain adaptation,
- active learning,
- self-organizing maps,
- and targeted spectroscopic follow-up.

A central challenge is ensuring that the spectroscopic training set adequately covers the photometric deployment manifold.

# Final Discussion

This notebook demonstrated several major ideas in machine learning for astrophysics:

- supervised regression,
- photometric redshifts,
- covariate shift,
- ensemble methods,
- uncertainty proxies,
- catastrophic outliers,
- and feature degeneracies.

Important limitations of this notebook:

- the mock catalogue is not physically realistic,
- the uncertainty estimates are not Bayesian,
- and the train/test split is simplified.

Real photometric-redshift pipelines often include:

- template fitting,
- hybrid template+ML approaches,
- nearest-neighbour methods,
- self-organizing maps,
- Gaussian processes,
- and deep learning.

One of the central lessons is:

A model can appear excellent on a biased validation set while failing badly in the true deployment distribution.